# Práctica — Clase 3: Clasificación, regresión y la Functional API de Keras

En esta práctica vas a completar dos ejercicios:

1. Adaptar el ejemplo de clasificación de cáncer de mama de `BreastCancer.ipynb` a TensorFlow 2.x, eligiendo tú la función de pérdida y la activación de salida correctas
2. Construir con la Functional API de Keras un modelo con dos entradas independientes

Completa las celdas marcadas con `# TODO`. Si te atascas, cada ejercicio tiene una solución colapsada justo debajo: intenta resolverlo antes de mirarla.

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

print("Versión de TensorFlow:", tf.__version__)

## Ejercicio 1 — Clasificar tumores de mama (basado en BreastCancer.ipynb)

El notebook original `BreastCancer.ipynb` entrena una red que decide si un
tumor es benigno o maligno a partir de 30 medidas numéricas (dataset de
Wisconsin), subiendo manualmente cuatro archivos CSV ya divididos y
normalizados. Aquí vas a reproducir la misma idea, pero cargando el dataset
directamente con `scikit-learn` y dejando que Keras normalice los datos.

**Pistas:**
- Es un problema de **clasificación binaria** (benigno o maligno): repasa la
  tabla del capítulo 05 del manual para elegir la función de pérdida y la
  activación de la última capa
- Usa `tf.keras.layers.Normalization` para normalizar las 30 características
  dentro del propio modelo, como en el ejemplo de Auto MPG del notebook de
  teoría
- La métrica adecuada para clasificación es `accuracy`, no `mean_absolute_error`

In [ ]:
# Cargamos el dataset de cáncer de mama de Wisconsin (incluido en scikit-learn)
datos = load_breast_cancer()
X, y = datos.data, datos.target
print("Forma de X:", X.shape)
print("Clases:", datos.target_names)  # 0 = maligno, 1 = benigno

# Dividimos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# TODO 1: crea una capa tf.keras.layers.Normalization y adáptala (.adapt()) a X_train
normalizador = None  # sustituye None por tu código

# TODO 2: construye un tf.keras.Sequential con la capa de normalización, dos capas
# ocultas Dense con activación 'relu' (por ejemplo 16 y 8 neuronas), y la capa de
# salida correcta para un problema de clasificación binaria
modelo_bc = None  # sustituye None por tu código

# TODO 3: compila el modelo con la función de pérdida y la métrica adecuadas
# para clasificación binaria (pista: 'binary_crossentropy' y 'accuracy')
# modelo_bc.compile(optimizer=..., loss=..., metrics=...)


In [ ]:
# Una vez completado el TODO anterior, entrena y evalúa el modelo
historial_bc = modelo_bc.fit(X_train, y_train, validation_split=0.2, epochs=30, verbose=0)

perdida, precision = modelo_bc.evaluate(X_test, y_test, verbose=0)
print(f"Pérdida en prueba: {perdida:.4f}")
print(f"Precisión en prueba: {precision:.2%}")

<details>
<summary>💡 Ver solución</summary>

```python
# Cargamos el dataset de cáncer de mama de Wisconsin (incluido en scikit-learn)
datos = load_breast_cancer()
X, y = datos.data, datos.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# La capa de normalización se adapta a las estadísticas (media y varianza) del
# conjunto de entrenamiento, nunca al conjunto de prueba
normalizador = tf.keras.layers.Normalization(axis=-1)
normalizador.adapt(X_train)

# Es clasificación binaria: la última capa tiene una neurona con activación
# sigmoid, que devuelve un número entre 0 y 1 interpretable como probabilidad
modelo_bc = tf.keras.Sequential([
    normalizador,
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# binary_crossentropy es la función de pérdida correcta para clasificación
# binaria; accuracy es la métrica que tiene sentido aquí (no mean_absolute_error)
modelo_bc.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'])

historial_bc = modelo_bc.fit(X_train, y_train, validation_split=0.2, epochs=30, verbose=0)

perdida, precision = modelo_bc.evaluate(X_test, y_test, verbose=0)
print(f"Pérdida en prueba: {perdida:.4f}")
print(f"Precisión en prueba: {precision:.2%}")
```
</details>

## Ejercicio 2 — Un modelo con dos entradas usando la Functional API

Imagina un sistema que evalúa una solicitud de préstamo combinando dos
fuentes de información:

- Datos **numéricos** del solicitante: ingresos, edad, historial (6 valores)
- Datos **categóricos** ya codificados en one-hot: tipo de contrato, región
  (5 valores)

Este modelo necesita **dos entradas independientes** que se combinan antes
de decidir la salida final (por ejemplo, la probabilidad de conceder el
préstamo). La Sequential API no puede expresar dos entradas, así que vas a
usar la Functional API, como en la sección 3 del notebook de teoría.

**Pistas:**
- Crea dos `tf.keras.Input`, uno con `shape=(6,)` y otro con `shape=(5,)`
- Procesa cada entrada con su propia capa `Dense` antes de combinarlas
- Une las dos ramas con `tf.keras.layers.concatenate([...])`
- La salida es una probabilidad (conceder o no el préstamo): piensa en qué
  activación y qué pérdida corresponden a ese tipo de problema

In [ ]:
# Datos de ejemplo (aleatorios) solo para comprobar que el modelo funciona
np.random.seed(0)
datos_numericos = np.random.random((200, 6)).astype("float32")
datos_categoricos = np.random.randint(2, size=(200, 5)).astype("float32")
etiquetas_prestamo = np.random.randint(2, size=(200, 1)).astype("float32")

# TODO 1: crea dos entradas con tf.keras.Input, una para los datos numéricos
# (shape=(6,)) y otra para los datos categóricos (shape=(5,))
entrada_numerica = None
entrada_categorica = None

# TODO 2: pasa cada entrada por al menos una capa Dense con activación 'relu'
rama_numerica = None
rama_categorica = None

# TODO 3: combina las dos ramas con tf.keras.layers.concatenate([...])
combinado = None

# TODO 4: añade la capa de salida con la activación adecuada para decidir
# "conceder" (1) o "no conceder" (0) el préstamo
salida = None

# TODO 5: crea el modelo con tf.keras.Model indicando las dos entradas y la salida
modelo_prestamo = None


In [ ]:
# Una vez completados los TODO anteriores, compila y entrena el modelo
# modelo_prestamo.compile(optimizer='adam', loss=..., metrics=['accuracy'])

# modelo_prestamo.fit(
#     {"datos_numericos": datos_numericos, "datos_categoricos": datos_categoricos},
#     etiquetas_prestamo,
#     epochs=5, verbose=1)

# modelo_prestamo.summary()


<details>
<summary>💡 Ver solución</summary>

```python
# Dos entradas independientes: una para los datos numéricos, otra para los categóricos
entrada_numerica = tf.keras.Input(shape=(6,), name="datos_numericos")
entrada_categorica = tf.keras.Input(shape=(5,), name="datos_categoricos")

# Cada entrada sigue su propio camino antes de combinarse
rama_numerica = tf.keras.layers.Dense(16, activation='relu')(entrada_numerica)
rama_numerica = tf.keras.layers.Dense(8, activation='relu')(rama_numerica)

rama_categorica = tf.keras.layers.Dense(8, activation='relu')(entrada_categorica)

# concatenate une las dos ramas en un único vector de características
combinado = tf.keras.layers.concatenate([rama_numerica, rama_categorica])

# Es una decisión binaria (conceder o no el préstamo): sigmoid + binary_crossentropy
salida = tf.keras.layers.Dense(1, activation='sigmoid')(combinado)

modelo_prestamo = tf.keras.Model(
    inputs=[entrada_numerica, entrada_categorica],
    outputs=salida,
    name="modelo_prestamo")

modelo_prestamo.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

modelo_prestamo.fit(
    {"datos_numericos": datos_numericos, "datos_categoricos": datos_categoricos},
    etiquetas_prestamo,
    epochs=5, verbose=1)

modelo_prestamo.summary()
```
</details>

## Para seguir practicando

- En el ejercicio 1, prueba a cambiar `mean_absolute_error` por
  `binary_crossentropy` a propósito y observa qué mensaje de error o qué
  comportamiento extraño produce entrenar una clasificación con una pérdida
  de regresión
- En el ejercicio 2, añade una tercera entrada (por ejemplo, un histórico de
  pagos de longitud 3) y combínala también en el `concatenate`
- Repasa el capítulo 05 del manual si alguna pista no te ha quedado clara